# Lab 11: Advanced Optimization (Gemini 3.1)

In this final lab of the series, we cover advanced features to improve performance, cost, and accuracy: **Context Caching**, **Embeddings**, and **Grounding**.

## Objectives
1. Implement **Context Caching** to reduce costs on long documents.
2. Generate **Text Embeddings** for semantic search.
3. Use **Google Search Grounding** for up-to-date information.
4. Explore **Multimodal Context Caching** for media files.

In [ ]:
import os

import httpx
import numpy as np
from dotenv import load_dotenv
from google import genai
from google.genai import types
from IPython.display import Markdown, display

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

print("Client initialized.")

## 1. Context Caching (Text)

Context caching allows you to store frequently used data on Google's servers. You pay for storage, but reuse is significantly cheaper.

In [ ]:
# 1. Create a large text (simulating a long document)
long_document = (
    "The history of the Galactic Empire started in the year 1254... " +
    ("A lot of detailed facts and lore... ") * 500
)

# 2. Create the cache (using gemini-3.1-flash-lite-preview)
print("Creating context cache...")
cache = client.caches.create(
    model="gemini-3.1-flash-lite-preview",
    config=types.CreateCachedContentConfig(
        contents=[long_document],
        ttl="3600s",
        display_name="galactic_lore_cache"
    )
)

print(f"Cache created: {cache.name}")

# 3. Use the cache
response = client.models.generate_content(
    model="gemini-3.1-flash-lite-preview",
    contents="What year did the Galactic Empire start?",
    config=types.GenerateContentConfig(cached_content=cache.name)
)

print(f"Response: {response.text}")

# Cleanup
client.caches.delete(name=cache.name)
print("Cache deleted.")

## 2. Text Embeddings

Representing text as numerical vectors for similarity search.

In [ ]:
sentences = [
    "The cat is sleeping on the mat.",
    "AI engineering is a growing field.",
    "Python is a versatile programming language."
]

embed_model = "gemini-embedding-001"
embeddings = client.models.embed_content(
    model=embed_model,
    contents=sentences
)
query_embedding = client.models.embed_content(
    model=embed_model,
    contents="animal"
)

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

similarities = [cosine_similarity(
    query_embedding.embeddings[0].values, e.values) for e in embeddings.embeddings]
best_match_idx = np.argmax(similarities)

print(
    f"Best match: '{sentences[best_match_idx]}' "
    f"(Score: {similarities[best_match_idx]:.4f})"
)

## 3. Google Search Grounding

Anchoring responses in real-time web results.

In [ ]:
prompt = "What is the latest news about the mission to Europa (Jupiter's moon)?"

response = client.models.generate_content(
    model="gemini-3.1-flash-lite-preview",
    config=types.GenerateContentConfig(
        tools=[types.Tool(google_search=types.GoogleSearch())]
    ),
    contents=prompt
)

display(Markdown(response.text))

## 4. Multimodal Context Caching

You can cache media files (audio/video) to perform multiple operations without paying for re-uploading and tokenizing the file each time.

In [ ]:
# 1. Upload a sample audio
audio_url = "https://storage.googleapis.com/generativeai-downloads/data/State_of_the_Union_Address_30_January_1961.mp3"
audio_data = httpx.get(audio_url).content
with open("temp_audio.mp3", "wb") as f:
    f.write(audio_data)
file_obj = client.files.upload(file="temp_audio.mp3")

# 2. Cache the media file
mm_cache = client.caches.create(
    model="gemini-3.1-flash-lite-preview",
    config=types.CreateCachedContentConfig(
        contents=[file_obj],
        ttl="600s"
    )
)

print(f"Multimodal cache created for: {file_obj.name}")

# Cleanup
client.caches.delete(name=mm_cache.name)
client.files.delete(name=file_obj.name)
os.remove("temp_audio.mp3")

## Summary

You have explored advanced optimization techniques:
1. **Caching**: Drastic cost reduction for persistent context (Text & Multimodal).
2. **Embeddings**: Foundation for semantic retrieval.
3. **Google Search Grounding**: Up-to-date information.